# The Mathematics of Ridge Regression

To understand how Ridge Regression physically shrinks weights, we modify the core Loss Function of Ordinary Least Squares (OLS) by adding a mathematical penalty.

An critical rule in regularized regression is that the intercept ($b$) is never penalized. Penalizing the intercept would mean our model's predictions change if we simply shift the baseline height of our target data ($y$), which ruins the physical meaning of the model. We only penalize the slopes (coefficients).

## 1. The 2D Formulation (Slope $m$, Intercept $b$)

In a simple 2D space with a single feature $x$, our prediction is $\hat{y}_i = mx_i + b$.

The standard OLS Loss Function is Mean Squared Error:

$$L_{OLS}(m, b) = \frac{1}{N} \sum_{i=1}^N (y_i - (mx_i + b))^2$$

Ridge Regression modifies this by appending the L2 Penalty Term (the squared value of the slope multiplied by our hyperparameter $\alpha$):

$$L_{Ridge}(m, b) = \frac{1}{N} \sum_{i=1}^N (y_i - (mx_i + b))^2 + \alpha m^2$$

### The Calculus Optimization in 2D

To see how $\alpha$ forces $m$ to shrink, we take the partial derivative of our new Ridge Loss Function with respect to the slope $m$:

$$\frac{\partial L_{Ridge}}{\partial m} = \left[ -\frac{2}{N} \sum_{i=1}^N x_i(y_i - \hat{y}_i) \right] + 2\alpha m$$

Notice what just happened. The first half of the equation inside the brackets is the exact derivative of standard OLS. The second half is the derivative of our penalty term ($2\alpha m$).

When we set this derivative to zero to find the minimum point of the loss bowl, the $+2\alpha m$ term acts as an extra mathematical drag. It actively forces the calculated value of $m$ to be smaller than what OLS would have chosen.

## 2. The Multi-Dimensional Matrix Derivation

When you scale to $m$ features, working with individual summation symbols becomes inefficient. We transition to matrix algebra.

Let:
*   $Y$ be an $N \times 1$ vector of true target values.
*   $X$ be an $N \times m$ matrix of scaled features.
*   $W$ be an $m \times 1$ vector of coefficients (weights).

The multi-dimensional Ridge Loss Function in matrix form is written as:

$$L(W) = (Y - XW)^T(Y - XW) + \alpha W^TW$$

Where $W^TW$ is the vector dot product $w_1^2 + w_2^2 + \dots + w_m^2$, yielding the total squared magnitude of all coefficients.

### Step-by-Step Matrix Derivation

First, we expand the OLS portion of the expression:

$$L(W) = Y^TY - 2W^TX^TY + W^TX^TXW + \alpha W^TW$$

Next, we take the derivative of the entire matrix equation with respect to the weight vector $W$:

$$\frac{\partial L}{\partial W} = -2X^TY + 2X^TXW + 2\alpha W$$

To find the optimal weights ($W_{Ridge}$), we set the vector gradient to zero:

$$-2X^TY + 2X^TXW + 2\alpha W = 0$$

Divide the entire equation by 2 and isolate the terms containing $W$:

$$X^TXW + \alpha W = X^TY$$

To factor out the weight vector $W$, we must multiply $\alpha$ by an Identity Matrix ($I$) of dimension $m \times m$ (a matrix with 1s down the main diagonal and 0s elsewhere). This preserves the matrix dimensions:

$$(X^TX + \alpha I)W = X^TY$$

Finally, to isolate $W$, we multiply both sides by the matrix inverse of $(X^TX + \alpha I)$:

$$W_{Ridge} = (X^TX + \alpha I)^{-1} X^TY$$

## 3. Comparing OLS vs. Ridge Closed-Form Solutions

Let's look at the two final mathematical engines side-by-side:

| Algorithm                    | Closed-Form Equation                |
| :--------------------------- | :---------------------------------- |
| Ordinary Least Squares (OLS) | $W_{OLS} = (X^TX)^{-1} X^TY$       |
| Ridge Regression (L2)        | $W_{Ridge} = (X^TX + \alpha I)^{-1} X^TY$ |

## Why this fixes the Multicollinearity Loophole

This matrix comparison reveals exactly why Ridge handles catastrophic feature correlation when OLS completely breaks.

When two features are highly correlated (e.g., trying to predict target values using both col_A and a duplicate col_B), columns in your feature matrix $X$ become linearly dependent. Geometrically, this causes the matrix product $X^TX$ to become singular or nearly singular.

A singular matrix has a determinant of exactly $0$, meaning it is mathematically impossible to invert. You cannot calculate $(X^TX)^{-1}$.

In code, if $X^TX$ is nearly singular, the inversion math blows up. Tiny variations in your data cause the calculated weights to swing wildly into millions or billions, destroying your model's stability (Extreme Variance).

### The Ridge Fix:

By adding $\alpha I$, Ridge manually adds a small positive constant $\alpha$ directly to the main diagonal of the $X^TX$ matrix before calculating the inverse:

$$X^TX + \alpha I = \begin{bmatrix} \\
x_1^1 & x_1^2 \\
x_2^1 & x_2^2 \\
\end{bmatrix} + \begin{bmatrix} \\
\alpha & 0 \\
0 & \alpha \\
\end{bmatrix}$$

This mathematical shift guarantees that the matrix $(X^TX + \alpha I)$ is strictly non-singular and always invertible, regardless of how correlated your features are. It completely stabilizes the internal matrix calculations.